# Project 1: Indian Orchards & Fruit Crops Yield & Productivity Diagnostic
## Phase 1: Exploratory Data Analysis & Data Cleaning (Pandas)

This notebook processes `india_fruit_apy_raw.csv` to clean text formatting, handle missing/invalid records, compute physical yield density (Tonnes/Ha), and assign productivity tiers across Indian fruit crops.

In [2]:
import pandas as pd
import numpy as np

# Task 1: Load and Inspect Raw Data
df_raw = pd.read_csv('india_fruit_apy_raw.csv')

print("=== RAW DATASET OVERVIEW ===")
print(f"Total Rows: {len(df_raw)}")
print("\nMissing Values Per Column:")
print(df_raw.isnull().sum())
print("\nFirst 5 Rows:")
df_raw.head()

=== RAW DATASET OVERVIEW ===
Total Rows: 670

Missing Values Per Column:
record_id             0
crop_year             0
state                 0
district              0
crop                  0
season                0
area_hectares         0
production_tonnes    12
dtype: int64

First 5 Rows:


,record_id,crop_year,state,district,crop,season,area_hectares,production_tonnes
0,1,2018,MAHARASHTRA,Nagpur,MANGO,Whole Year,13490.14,136983.46
1,2,2018,Maharashtra,Nashik,Mango,Whole Year,13943.07,199491.51
2,3,2018,Maharashtra,Jalgaon,Mango,Whole Year,11297.54,112011.24
3,4,2018,Maharashtra,Ratnagiri,Mango,Whole Year,16737.64,207857.84
4,5,2018,Maharashtra,Ahmednagar,MANGO,Whole Year,10591.58,125578.01


## Task 2: Deduplication & Text Normalization
We remove duplicate records and standardize state, district, and crop names to eliminate leading/trailing spaces and capitalization inconsistencies.

In [3]:
# Drop duplicate rows
initial_rows = len(df_raw)
df_clean = df_raw.drop_duplicates().copy()
print(f"Removed {initial_rows - len(df_clean)} duplicate rows.")

# Standardize text columns (strip whitespace and title case)
text_cols = ['state', 'district', 'crop', 'season']
for col in text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.title()

print("\nCleaned Unique Crops:", df_clean['crop'].unique())
print("Cleaned Unique States:", df_clean['state'].unique())

Removed 10 duplicate rows.

Cleaned Unique Crops: ['Mango' 'Apple' 'Banana' 'Citrus' 'Grapes' 'Guava']
Cleaned Unique States: ['Maharashtra' 'Andhra Pradesh' 'Uttar Pradesh' 'Karnataka' 'Gujarat'
 'Jammu & Kashmir' 'Himachal Pradesh']


## Task 3: Data Hygiene (Handling Missing & Invalid Values)
We remove records with invalid cultivated area (<= 0 Hectares) or missing production values to ensure accurate yield calculations.

In [4]:
# Filter out invalid area and missing production
valid_area_mask = df_clean['area_hectares'] > 0
valid_prod_mask = df_clean['production_tonnes'].notna()

df_clean = df_clean[valid_area_mask & valid_prod_mask].copy()
print(f"Final Verified Clean Records: {len(df_clean)} rows")

Final Verified Clean Records: 645 rows


## Task 4: Feature Engineering (Yield Calculation & Productivity Tiering)
We engineer physical yield density using the formula:
$$\text{Yield (Tonnes/Ha)} = \frac{\text{Production (Tonnes)}}{\text{Area (Hectares)}}$$
And assign each record to a crop-specific productivity tier.

In [5]:
# Calculate Yield in Tonnes per Hectare
df_clean['yield_tonnes_per_ha'] = df_clean['production_tonnes'] / df_clean['area_hectares']

# Define Tiering Logic
def assign_productivity_tier(row):
    crop = row['crop']
    y = row['yield_tonnes_per_ha']

    benchmarks = {
        'Banana': (45, 35),
        'Grapes': (25, 18),
        'Guava': (16, 12),
        'Citrus': (15, 11),
        'Apple': (13, 9),
        'Mango': (11, 8)
    }

    high, mod = benchmarks.get(crop, (15, 10))
    if y >= high:
        return 'High Yield'
    elif y >= mod:
        return 'Moderate Yield'
    else:
        return 'Low Yield'

df_clean['productivity_tier'] = df_clean.apply(assign_productivity_tier, axis=1)

print("Productivity Tier Breakdown:")
print(df_clean['productivity_tier'].value_counts())

Productivity Tier Breakdown:
productivity_tier
High Yield        282
Moderate Yield    261
Low Yield         102
Name: count, dtype: int64


## Task 5: Summary Statistics & Export
We calculate total area, total production, and average yield by crop, then export the clean dataset.

In [6]:
# Group by crop to check summary metrics
crop_summary = df_clean.groupby('crop').agg(
    total_area_ha=('area_hectares', 'sum'),
    total_production_tonnes=('production_tonnes', 'sum'),
    avg_yield_t_ha=('yield_tonnes_per_ha', 'mean')
).reset_index()

print("=== CROP SUMMARY TABLE ===")
print(crop_summary.round(2).to_string(index=False))

# Export clean CSV
df_clean.to_csv('india_fruit_apy_clean.csv', index=False)
print("\nSaved clean dataset as 'india_fruit_apy_clean.csv'!")

=== CROP SUMMARY TABLE ===
  crop  total_area_ha  total_production_tonnes  avg_yield_t_ha
 Apple      840393.81              10121947.15           12.06
Banana      935551.50              42192251.80           45.17
Citrus     1141796.38              15962621.86           13.98
Grapes      559777.32              13667964.86           24.54
 Guava      579866.73               8977272.02           15.36
 Mango     1802681.77              18631090.73           10.41

Saved clean dataset as 'india_fruit_apy_clean.csv'!
